# Project 1 — Problem Solution Pipeline

Canonical notebook for DS 4320 Project 1: load relational D1 CSVs into **DuckDB**, run SQL joins, fit a **linear regression** baseline for next-day FAANG returns, and plot coefficients.

Data lives in [`../data/`](../data/) (four tables: `tickers`, `daily_prices`, `daily_features`, `calendar`). Regenerate CSVs with [`../data/build_project1_data.py`](../data/build_project1_data.py).

## Data Creation (rubric §8.2)

### Item 1 — Provenance

Project 1 dataset for next-day FAANG return modeling from public daily data on Stooq (https://stooq.com). Daily OHLCV for META, AAPL, AMZN, NFLX, GOOGL over 2019-01-01 to 2024-12-31, standardized into D1 with four CSV tables under `Project1/data/`.

### Items 3–5 — Bias, mitigation, judgement

See course README in `Project1/README.md` for full text (bias: FAANG-only slice, regime shifts, vendor conventions; mitigation: time-based split, traceable keys, cautious interpretation; uncertainty: non-stationarity and feature choices).

## Metadata (rubric §8.2)

Logical schema: `tickers` → `daily_prices` → `daily_features`; `calendar` keyed by `trade_date`. Full data dictionary and numerical-uncertainty table are in `Project1/README.md`.

## Pipeline Check — Item 1

Load all four CSVs into DuckDB, join model-ready rows, apply a strict date split (train &lt; 2024-01-01, test ≥ 2024-01-01), train `LinearRegression`, report MAE / RMSE / R².

In [ ]:
from pathlib import Path

import duckdb
import matplotlib.pyplot as plt
import pandas as pd
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score


def resolve_data_dir() -> Path:
    """Find Project1/data whether cwd is repo root, Project1/, or pipeline/."""
    here = Path.cwd().resolve()
    candidates = [
        here / "data",
        here / "Project1" / "data",
        here.parent / "data",
    ]
    for p in candidates:
        if (p / "tickers.csv").is_file():
            return p
    raise FileNotFoundError(
        "Could not find Project1/data/tickers.csv. Run this notebook from DS4320, Project1, or Project1/pipeline."
    )


DATA_DIR = resolve_data_dir()
print("DATA_DIR:", DATA_DIR)

### DuckDB: register CSVs and sample SQL

Demonstrates relational queries on D1 before modeling.

In [ ]:
con = duckdb.connect()
d = DATA_DIR.as_posix()
con.execute(f"""
CREATE OR REPLACE TABLE tickers AS SELECT * FROM read_csv_auto('{d}/tickers.csv', HEADER=TRUE);
CREATE OR REPLACE TABLE daily_prices AS SELECT * FROM read_csv_auto('{d}/daily_prices.csv', HEADER=TRUE);
CREATE OR REPLACE TABLE daily_features AS SELECT * FROM read_csv_auto('{d}/daily_features.csv', HEADER=TRUE);
CREATE OR REPLACE TABLE calendar AS SELECT * FROM read_csv_auto('{d}/calendar.csv', HEADER=TRUE);
""")

con.execute("""
SELECT t.symbol, COUNT(*) AS n_days,
       MIN(p.trade_date) AS first_date,
       MAX(p.trade_date) AS last_date
FROM daily_prices p
JOIN tickers t ON t.ticker_id = p.ticker_id
GROUP BY t.symbol
ORDER BY t.symbol
""").fetch_df()

### Join features and fit baseline model

In [ ]:
rows_df = con.execute("""
SELECT f.trade_date, t.symbol,
       f.return_1d, f.volatility_5d, f.ma_5_dev, f.ma_20_dev, f.volume_pct_change,
       f.target_return_next_day
FROM daily_features f
JOIN tickers t ON t.ticker_id = f.ticker_id
WHERE f.target_return_next_day IS NOT NULL
  AND f.return_1d IS NOT NULL
  AND f.volatility_5d IS NOT NULL
  AND f.ma_5_dev IS NOT NULL
  AND f.ma_20_dev IS NOT NULL
  AND f.volume_pct_change IS NOT NULL
ORDER BY f.trade_date, t.symbol
""").fetch_df()

rows_df["trade_date"] = pd.to_datetime(rows_df["trade_date"])
train_df = rows_df[rows_df["trade_date"] < "2024-01-01"].copy()
test_df = rows_df[rows_df["trade_date"] >= "2024-01-01"].copy()

feature_cols = ["return_1d", "volatility_5d", "ma_5_dev", "ma_20_dev", "volume_pct_change"]
X_train = train_df[feature_cols]
y_train = train_df["target_return_next_day"]
X_test = test_df[feature_cols]
y_test = test_df["target_return_next_day"]

model = LinearRegression()
model.fit(X_train, y_train)
preds = model.predict(X_test)

print("Rows (joined feature rows):", len(rows_df))
print("Train rows:", len(train_df), "Test rows:", len(test_df))
print("MAE:", round(mean_absolute_error(y_test, preds), 6))
print("RMSE:", round(mean_squared_error(y_test, preds) ** 0.5, 6))
print("R2:", round(r2_score(y_test, preds), 6))

coef_df = pd.DataFrame({"feature": feature_cols, "coefficient": model.coef_})
coef_df = coef_df.sort_values("coefficient", key=lambda s: s.abs(), ascending=False)
coef_df

## Pipeline Check — Item 2

Coefficient bar chart (out-of-sample year 2024).

In [ ]:
plot_df = coef_df.copy().sort_values("coefficient")
colors = ["steelblue" if c >= 0 else "coral" for c in plot_df["coefficient"]]

fig, ax = plt.subplots(figsize=(9, 5))
bars = ax.barh(plot_df["feature"], plot_df["coefficient"], color=colors, edgecolor="black")
ax.axvline(0, color="black", linewidth=0.8)
ax.set_xlabel("Linear Regression Coefficient")
ax.set_title("Project 1 Baseline Model: Feature Coefficients (Tested on 2024 Data)")

for bar, val in zip(bars, plot_df["coefficient"]):
    ax.text(
        val + (0.0005 if val >= 0 else -0.0005),
        bar.get_y() + bar.get_height() / 2,
        f"{val:+.4f}",
        va="center",
        ha="left" if val >= 0 else "right",
        fontsize=9,
    )

plt.tight_layout()
plt.show()